In [ ]:
import os
import sys
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import auc

sys.path.append(os.path.abspath("../"))

from util import dataset as u_dataset

In [ ]:
def read_csv_file(file_path):
    """
    Reads a CSV file from the given path and returns a pandas DataFrame.

    Parameters:
    file_path (str): Path to the CSV file.

    Returns:
    pd.DataFrame: DataFrame containing the CSV data.
    """
    try:
        df = pd.read_csv(file_path)
        return df
    except FileNotFoundError:
        print(f"Error: The file at {file_path} was not found.")
        return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None


classifier_basic = read_csv_file("../../data/evaluation/classifier-basic/run_1.csv")

ceilings = {
    u_dataset.IntersectionType.L.name: float(
        classifier_basic[classifier_basic["architecture"] == "conv_v1"][
            "recall_per_l_intersection"
        ].iloc[0]
    ),
    u_dataset.IntersectionType.T.name: float(
        classifier_basic[classifier_basic["architecture"] == "conv_v1"][
            "recall_per_t_intersection"
        ].iloc[0]
    ),
    u_dataset.IntersectionType.X.name: float(
        classifier_basic[classifier_basic["architecture"] == "conv_v1"][
            "recall_per_x_intersection"
        ].iloc[0]
    ),
}

In [ ]:
log_path = Path("../../logs/fit/")

In [ ]:
architectures = architectures = [
    r"$\mathrm{Cla}_1$",
    r"$\mathrm{Cla}_2$",
    r"$\mathrm{Cla}_3$",
    r"$\mathrm{Cla}_4$",
]

classes = ["L-Kreuzung", "T-Kreuzung", "X-Kreuzung"]

In [ ]:
# --- Constants ---
log_path = Path("../../logs/fit/")
architectures = [r"$\mathrm{Cla}_1$", r"$\mathrm{Cla}_2$", r"$\mathrm{Cla}_3$", r"$\mathrm{Cla}_4$"]
specifications = ["v1", "v2", "v3", "v4"]

classes = ["L-Kreuzung", "T-Kreuzung", "X-Kreuzung"]
runs = [1, 2, 3]  # Assuming runs are 1, 2, 3


# --- Data Extraction ---
def extract_ap_values():
    ap_values = {
        cls.name: [[] for _ in range(len(specifications))]
        for cls in list(u_dataset.IntersectionType)[1:]
    }
    ceiling_values = {
        cls.name: [[] for _ in range(len(specifications))]
        for cls in list(u_dataset.IntersectionType)[1:]
    }
    ceiling_values_mean = {
        cls.name: [[] for _ in range(len(specifications))]
        for cls in list(u_dataset.IntersectionType)[1:]
    }

    for run in runs:
        for arch_idx, arch in enumerate(specifications):
            for cls in list(u_dataset.IntersectionType)[1:]:
                base_dir = Path(log_path, "classifier-basic", f"run_{run}", arch)
                folders = [f for f in base_dir.iterdir() if f.is_dir()]
                if len(folders) == 1:
                    path_to_model = folders[0]
                else:
                    raise ValueError(
                        f"Expected exactly one folder in {base_dir}, found {len(folders)}"
                    )

                # Load AP values for the class
                precisions = np.load(
                    path_to_model.as_posix() + f"/precisions/intersections_{cls.name}.npy"
                )
                recalls = np.load(
                    path_to_model.as_posix() + f"/recalls/intersections_{cls.name}.npy"
                )
                ap_values[cls.name][arch_idx].append(auc(recalls, precisions))
                ceiling_values[cls.name][arch_idx].append(np.max(recalls))

    for cls in list(u_dataset.IntersectionType)[1:]:
        ceiling_values_mean[cls.name] = np.max(ceiling_values[cls.name], 1)
        print("Std:", np.std(ceiling_values[cls.name], 1))

    print(ceiling_values_mean)
    return ap_values, ceiling_values_mean


# --- Plotting ---
def plot_ap_values(ap_values, ceiling_values):
    x = np.arange(len(architectures))
    n_classes = len(classes)
    bar_width = 0.22
    offsets = np.linspace(-(n_classes - 1) / 2, (n_classes - 1) / 2, n_classes) * bar_width
    colors = ["#56B4E9", "#D55E00", "#CC79A7"]  # Okabe-Ito

    fig, ax = plt.subplots(figsize=(9, 5))

    for cls, color, offset in zip(
        list(u_dataset.IntersectionType)[1:], colors, offsets, strict=True
    ):
        means = []
        for i, samples in enumerate(ap_values[cls.name]):
            samples = np.array(samples)
            mean = np.mean(samples)
            means.append(mean)

            # Plot individual points (jittered)
            jitter = np.linspace(-0.06, 0.06, len(samples))
            ax.scatter(
                x[i] + offset + jitter,
                samples,
                color=color,
                s=30,
                zorder=5,
                alpha=0.7,
                edgecolors="white",
                linewidths=0.4,
            )

            # Plot mean line
            ax.hlines(
                mean,
                x[i] + offset - 0.09,
                x[i] + offset + 0.09,
                colors=color,
                linewidth=2.5,
                zorder=6,
            )

            # ceiling = ceiling_values[cls.name]
            # ax.plot(
            #     x + offset,
            #     ceiling,
            #     color=color,
            #     linewidth=1.2,
            #     linestyle="--",
            #     alpha=0.8,
            #     zorder=2,
            # )
            ceiling = ceilings[cls.name]
            ax.axhline(
                y=ceiling,
                color=color,
                linewidth=1.2,
                linestyle="--",
                alpha=0.8,
                zorder=2,
            )

        # Connect the means for this class
        ax.plot(
            x + offset,
            means,
            color=color,
            linewidth=0.7,
            linestyle="-",
            zorder=4,
            alpha=0.9,
        )

    # --- Styling ---
    ax.set_xticks(x)
    ax.set_xticklabels(architectures, fontsize=11)
    ax.set_xlabel("Architektur", fontsize=11)
    ax.set_ylabel("AP", fontsize=11)
    ax.set_ylim(0.4, 1.05)
    ax.yaxis.grid(True, linestyle="--", alpha=0.6, zorder=0)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    # draw_ceilings(ax, ceiling_values)

    # --- Legend ---
    handles = [
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=color, markersize=7, label=cls)
        for cls, color in zip(classes, colors, strict=True)
    ]

    ceiling_handles = [
        plt.Line2D(
            [0],
            [0],
            color=color,
            linewidth=1.2,
            linestyle="--",
            alpha=0.8,
            label=r"Recall$@K_c$ für " + f"{cls}",
        )
        for cls, color in zip(classes, colors, strict=True)
    ]
    handles += ceiling_handles

    ax.legend(
        handles=handles,
        loc="lower right",
        framealpha=0.9,
        fontsize=9,
        ncols=2,
    )

    plt.tight_layout()
    plt.savefig("../../plots/classifier-basic/intersection_APs.pdf")
    plt.show()


# --- Execute ---
ap_values, ceiling_values = extract_ap_values()
plot_ap_values(ap_values, ceiling_values)